# Topic 06 — Merge / Join / Concat

> **Investigation milestone:** The truth lives *across* tables. To compute real
> profit you need order_items × products (cost), orders (channel, date),
> customers (segment), returns (refunds). One bad join can silently double your
> revenue — so this topic is about joining *safely*.

**Time split: 20% reading · 80% in `practice.ipynb`. NumPy recall checkpoint at the end.**

---

## `merge` — the SQL join of Pandas

```python
oi = items.merge(products, on="product_id", how="left")
```

### `how=` decides who survives
| how | keeps |
|---|---|
| `inner` | only keys in **both** (default) |
| `left` | all left rows; unmatched right → NaN |
| `right` | all right rows |
| `outer` | union of keys |

> A `left` join should **not** change your left row count — *unless* the right
> side has duplicate keys, which causes a **many-to-many blow-up** that inflates
> totals. This is the #1 way analysts corrupt revenue numbers.

### Guardrails (use them every time)
```python
items.merge(products, on="product_id", how="left",
            validate="many_to_one",   # raises if right keys aren't unique
            indicator=True)            # adds _merge: left_only/right_only/both
```
- `validate=` catches unexpected duplication *before* it ruins your sums.
- `indicator=True` lets you count `left_only` (orphans) and `right_only`.

### Different key names
```python
a.merge(b, left_on="cust", right_on="customer_id", how="left")
```

## `concat` — stacking, not matching

```python
pd.concat([jan, feb, mar], ignore_index=True)   # stack rows (axis=0)
pd.concat([df1, df2], axis=1)                    # glue columns (aligns on index!)
```
Use `concat` for "more of the same rows"; use `merge` for "bring in related columns".

## `join` — merge on the index

`a.join(b)` is `merge` that defaults to the index. Convenient when your tables
are already indexed by the key.

## Detecting damage
```python
before = len(items)
m = items.merge(products, on="product_id", how="left")
assert len(m) == before, "row count changed → many-to-many!"
```

## NumPy connection 🔢

Joins are *relational*, not array math — this is where Pandas goes beyond NumPy.
But the **validation** is pure NumPy thinking: `m["_merge"].value_counts()`,
`mask.sum()` to count orphans, set logic on key arrays. After a join you often
fill the NaNs from unmatched rows with `np.where`/`fillna`.

## Visual learning 📊

Post-join, a stacked bar of `_merge` categories (`both` vs `left_only`) instantly
shows your match rate — a data-quality chart worth keeping.

---

## 🔎 Interview Lens (answer in writing)
- **Q18:** Walk through inner/left/right/outer with two Aurora tables — who's lost or invented?
- **Q19:** What risks exist when merging? How do you detect a many-to-many blow-up *before* it corrupts totals?

### Recap
1. Why can a `left` join increase your row count?
2. What do `validate=` and `indicator=` protect you from?
3. `merge` vs `concat` in one line each.

➡️ Open **`practice.ipynb`** (includes NumPy recall checkpoint 2).